# MostarGate — Dataset & Validation Analysis

This notebook loads all generated data and metric files, displays summary statistics,
and renders the five paper-quality charts specified in `docs/paper-updates.md`.

**Run order:** This notebook is read-only — it loads files generated by the pipeline.
Run the pipeline first if the files are missing:
```
make dataset-stats        # dataset statistics
make validate-compare     # pre/post review metrics + disagreements.json
make validate-review      # interactive review (must be complete for post-review metrics)
```

**Figures** are saved as PDFs to `dataset/metrics/` for direct inclusion in the paper.

---

## Contents
1. Setup & data loading
2. Dataset statistics
3. Figure 1 — Dataset composition & grant rates
4. Validation metrics summary
5. Figure 2 — Pre-review vs post-review comparison
6. Figure 3 — Per-tool precision & recall by tier
7. Figure 4 — Disagreement resolution breakdown
8. Figure 5 — Sensitivity confusion matrix
9. Key findings summary

## 1. Setup & data loading

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
from pathlib import Path

# ── Paths (notebook lives in notebooks/, data in dataset/) ──────────────────
ROOT    = Path("..").resolve()
METRICS = ROOT / "dataset" / "metrics"
METRICS.mkdir(parents=True, exist_ok=True)

# ── Colorblind-safe Okabe-Ito palette ───────────────────────────────────────
C = {
    "blue":      "#0072B2",
    "sky":       "#56B4E9",
    "green":     "#009E73",
    "orange":    "#E69F00",
    "vermillion":"#D55E00",
    "purple":    "#CC79A7",
    "yellow":    "#F0E442",
    "grey":      "#999999",
}
TIER_COLOR = {1: C["vermillion"], 2: C["orange"], 3: C["green"]}
TIER_LABEL = {1: "T1 — Default Deny", 2: "T2 — Grant w/ Justification", 3: "T3 — Default Permit"}

# ── Global chart style ──────────────────────────────────────────────────────
plt.rcParams.update({
    "figure.dpi":       150,
    "font.size":        10,
    "axes.spines.top":  False,
    "axes.spines.right":False,
    "axes.grid":        False,
})

# ── Tool metadata (mirrors constants.py) ────────────────────────────────────
TOOLS = [
    "github_read", "pull_request_create", "code_execute", "confluence_read",
    "jira_read", "jira_write", "slack_read", "slack_write", "salesforce_read",
    "database_read", "email_read", "email_send_external", "http_request",
    "file_read_uploaded", "export_file",
]
TOOL_TIER = {
    "database_read": 1, "email_send_external": 1,
    "http_request": 1,  "pull_request_create": 1,
    "github_read": 2,   "code_execute": 2,  "slack_read": 2,
    "slack_write": 2,   "jira_write": 2,    "salesforce_read": 2,
    "email_read": 2,    "export_file": 2,
    "confluence_read": 3, "jira_read": 3,   "file_read_uploaded": 3,
}
DEPTS = [
    "Customer Success", "Data and Analytics", "Engineering",
    "Finance", "Legal and Compliance", "Security",
]
SENSITIVITIES = ["LOW", "MEDIUM", "HIGH"]

print("Setup complete.")

In [ ]:
# ── Load data files ──────────────────────────────────────────────────────────
dataset       = json.loads((ROOT / "dataset/dataset.json").read_text())
human_results = json.loads((ROOT / "dataset/validation_results.json").read_text())
disagreements = json.loads((ROOT / "dataset/disagreements.json").read_text())
pre           = json.loads((METRICS / "metrics_pre_review.json").read_text())
post          = json.loads((METRICS / "metrics_post_review.json").read_text())

print(f"Dataset records:           {len(dataset)}")
print(f"Human-validated records:   {len(human_results)}")
print(f"Records with disagreements:{len(disagreements)}")
print(f"Pre-review generated:      {pre['meta']['generated_at'][:10]}")
print(f"Post-review generated:     {post['meta']['generated_at'][:10]}")

## 2. Dataset statistics

Overview of the 600-record synthetic dataset generated from the TechCorp company policy.

In [ ]:
from collections import Counter

n = len(dataset)
dept_counts  = Counter(r["department"] for r in dataset)
sens_counts  = Counter(r["sensitivity"] for r in dataset)
grant_rates  = {t: sum(1 for r in dataset if r["permissions"].get(t)) / n for t in TOOLS}
avg_perms    = sum(sum(1 for v in r["permissions"].values() if v) for r in dataset) / n

print(f"Total records:      {n}")
print(f"Avg permissions/record: {avg_perms:.2f}\n")

print(f"{'Department':<26}  {'Count':>5}  {'%':>5}")
print("-" * 40)
for d in sorted(DEPTS, key=lambda x: -dept_counts[x]):
    print(f"  {d:<24}  {dept_counts[d]:>5}  {dept_counts[d]/n*100:>4.0f}%")

print(f"\n{'Sensitivity':<12}  {'Count':>5}  {'%':>5}")
print("-" * 25)
for s in SENSITIVITIES:
    print(f"  {s:<10}  {sens_counts[s]:>5}  {sens_counts[s]/n*100:>4.0f}%")

print(f"\n{'Tool':<26} {'Tier':>4}  {'Grant rate':>10}")
print("-" * 45)
for t in sorted(TOOLS, key=lambda x: (TOOL_TIER[x], -grant_rates[x])):
    print(f"  {t:<24} T{TOOL_TIER[t]:>1}    {grant_rates[t]*100:>7.1f}%")

## 3. Figure 1 — Dataset composition & permission grant rates

**What this shows:** Two panels describing the shape of the dataset.
The left panel shows how many records belong to each department.
The right panel shows how often the LLM granted each permission across all 600 records,
coloured by tier.

**How to read the right panel:**
Bars are sorted by tier then by grant rate within tier.
Tier 1 tools (red) cluster at the left with very low grant rates — they are Default Deny
and the LLM rarely grants them.
Tier 3 tools (green) have the highest grant rates — they are Default Permit.

**Key finding:** There is significant class imbalance. Tier 1 tools like `http_request`,
`email_send_external`, and `database_read` are granted in fewer than 20% of records.
This means per-tool precision and recall figures for these tools are estimated from a
small number of positive examples — interpret their confidence intervals accordingly.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# ── Left: department distribution ───────────────────────────────────────────
depts_sorted = sorted(DEPTS, key=lambda d: dept_counts[d])
counts = [dept_counts[d] for d in depts_sorted]
bars = ax1.barh(depts_sorted, counts, color=C["blue"], alpha=0.82)
ax1.set_xlabel("Number of records")
ax1.set_title("Records by department", fontweight="bold")
for bar, c in zip(bars, counts):
    ax1.text(c + 3, bar.get_y() + bar.get_height() / 2,
             f"{c} ({c/n*100:.0f}%)", va="center", fontsize=9)
ax1.set_xlim(0, max(counts) * 1.25)

# ── Right: grant rate per tool by tier ──────────────────────────────────────
tools_sorted = sorted(TOOLS, key=lambda t: (TOOL_TIER[t], -grant_rates[t]))
rates  = [grant_rates[t] * 100 for t in tools_sorted]
colors = [TIER_COLOR[TOOL_TIER[t]] for t in tools_sorted]

bars2 = ax2.barh(tools_sorted, rates, color=colors, alpha=0.82)
ax2.set_xlabel("Grant rate (%)")
ax2.set_title("LLM permission grant rate by tool", fontweight="bold")

# Tier separators
current_tier = TOOL_TIER[tools_sorted[0]]
for i, t in enumerate(tools_sorted):
    if TOOL_TIER[t] != current_tier:
        ax2.axhline(y=i - 0.5, color="#cccccc", linewidth=1)
        current_tier = TOOL_TIER[t]

legend_patches = [
    mpatches.Patch(color=TIER_COLOR[k], label=TIER_LABEL[k]) for k in [1, 2, 3]
]
ax2.legend(handles=legend_patches, loc="lower right", fontsize=8)
ax2.set_xlim(0, 100)

plt.suptitle("Figure 1 — Dataset composition and LLM permission grant rates (n=600)",
             fontsize=11, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig(METRICS / "fig1_dataset_composition.pdf", bbox_inches="tight")
plt.show()
print("Saved → dataset/metrics/fig1_dataset_composition.pdf")

## 4. Validation metrics summary

Side-by-side table of all key metrics before and after disagreement adjudication.

**Pre-review:** human labels treated as ground truth.
**Post-review:** adjudicated labels as ground truth; ambiguous cases excluded.
The difference between the two rows quantifies the effect of human labelling errors.

In [ ]:
pre_ov  = pre["permission_metrics"]["overall"]
post_ov = post["permission_metrics"]["overall"]

rows = [
    ("Records evaluated",           "n_records",                   ""),
    ("Exact match rate",            "exact_match_rate",            ".1%"),
    ("Hamming accuracy",            "hamming_accuracy",            ".1%"),
    ("Macro F1",                    "macro_f1",                    ".3f"),
    ("Cohen's kappa",               "kappa",                       ".3f"),
    ("Overshoot rate",              "overshoot_rate",              ".1%"),
    ("Undershoot rate",             "undershoot_rate",             ".1%"),
    ("Severity-weighted overshoot", "severity_weighted_overshoot", ".2f"),
]

print(f"{'Metric':<33}  {'Pre-review':>12}  {'Post-review':>12}  {'Delta':>8}")
print("-" * 72)
for label, key, fmt in rows:
    pv  = pre_ov.get(key)
    pov = post_ov.get(key)
    if fmt == "":
        print(f"  {label:<31}  {str(pv):>12}  {str(pov):>12}")
    else:
        delta  = pov - pv
        sign   = "+" if delta >= 0 else ""
        pv_s   = format(pv,    fmt)
        pov_s  = format(pov,   fmt)
        del_s  = f"{sign}{format(delta, fmt)}"
        print(f"  {label:<31}  {pv_s:>12}  {pov_s:>12}  {del_s:>8}")

print()
rb = post["resolution_breakdown"]
print("Resolution breakdown (permissions):")
print(f"  LLM correct:   {rb['permissions']['llm_correct']}  (50%)")
print(f"  Human correct: {rb['permissions']['human_correct']}  (40%)")
print(f"  Ambiguous:     {rb['permissions']['ambiguous']}  (10%)")

## 5. Figure 2 — Pre-review vs post-review permission metrics

**What this shows:** Every key metric side by side before and after adjudication.
The right panel isolates severity-weighted overshoot on its own scale because
its value (15 vs 2) dwarfs the 0–1 range of the other metrics.

**How to read it:** Taller post-review bars are better for accuracy metrics
(kappa, F1, exact match). Shorter post-review bars are better for error metrics
(overshoot, undershoot). All post-review bars move in the correct direction.

**Key finding:** The severity-weighted overshoot drops from **15.00 to 2.00**
(86.7% reduction). This is the paper's headline security result.
The 15.00 pre-review figure was almost entirely human labelling errors —
cases where the human incorrectly denied a high-tier tool the LLM correctly granted.
The LLM's true over-granting of dangerous tools is near-zero.

The dashed line at κ = 0.80 marks the standard threshold for substantial agreement.
Both pre-review (0.917) and post-review (0.967) clear it comfortably.

In [ ]:
metric_keys   = ["exact_match_rate", "hamming_accuracy", "macro_f1",
                 "kappa", "overshoot_rate", "undershoot_rate"]
metric_labels = ["Exact Match", "Hamming Acc.", "Macro F1",
                 "Cohen's κ", "Overshoot", "Undershoot"]

pre_vals  = [pre_ov[k]  for k in metric_keys]
post_vals = [post_ov[k] for k in metric_keys]

x     = np.arange(len(metric_keys))
width = 0.35

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5),
                                gridspec_kw={"width_ratios": [3, 1]})

# ── Left: main metrics ───────────────────────────────────────────────────────
b1 = ax1.bar(x - width / 2, pre_vals,  width, label="Pre-review",
             color=C["sky"],  alpha=0.85)
b2 = ax1.bar(x + width / 2, post_vals, width, label="Post-review",
             color=C["blue"], alpha=0.90)

ax1.set_xticks(x)
ax1.set_xticklabels(metric_labels, rotation=18, ha="right")
ax1.set_ylabel("Score (0–1)")
ax1.set_ylim(0, 1.12)
ax1.set_title("Permission metrics — pre vs post-review", fontweight="bold")
ax1.axhline(y=0.80, color=C["grey"], linestyle="--", linewidth=0.9,
            label="κ = 0.80 threshold")
ax1.legend(fontsize=9)

for bar in list(b1) + list(b2):
    ax1.text(bar.get_x() + bar.get_width() / 2,
             bar.get_height() + 0.012,
             f"{bar.get_height():.3f}",
             ha="center", va="bottom", fontsize=7)

# ── Right: severity-weighted overshoot ──────────────────────────────────────
sev_pre  = pre_ov["severity_weighted_overshoot"]
sev_post = post_ov["severity_weighted_overshoot"]

ax2.bar(["Pre", "Post"], [sev_pre, sev_post],
        color=[C["sky"], C["blue"]], alpha=0.85, width=0.4)
ax2.set_title("Severity-weighted\novershoot", fontweight="bold")
ax2.set_ylabel("Weighted score")

for i, v in enumerate([sev_pre, sev_post]):
    ax2.text(i, v + 0.3, f"{v:.2f}", ha="center", va="bottom",
             fontweight="bold", fontsize=11)

reduction = (sev_pre - sev_post) / sev_pre * 100
ax2.annotate("",
             xy=(1, sev_post + 0.5), xytext=(0, sev_pre - 0.5),
             arrowprops=dict(arrowstyle="->", color=C["vermillion"], lw=1.5))
ax2.text(0.5, (sev_pre + sev_post) / 2,
         f"−{reduction:.0f}%",
         ha="center", color=C["vermillion"], fontweight="bold", fontsize=11)

plt.suptitle("Figure 2 — Pre-review vs post-review permission metrics (n=60)",
             fontsize=11, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig(METRICS / "fig2_pre_post_comparison.pdf", bbox_inches="tight")
plt.show()
print("Saved → dataset/metrics/fig2_pre_post_comparison.pdf")

## 6. Figure 3 — Per-tool precision & recall by tier

**What this shows:** For each of the 15 tools, the classifier's precision (solid bar)
and recall (lighter bar) using adjudicated post-review labels as ground truth.
Tools are grouped by tier with a divider between groups, and sorted by F1 within
each tier.

**How to read it:**
- **Precision** — when the LLM grants this tool, how often is that correct?
- **Recall** — when the tool should be granted, how often does the LLM catch it?
- **N/A** — tool has zero positive instances in the 60-record sample; metric is undefined.
  This is a sample-size artefact, not a classifier failure.

**Key findings:**
- Tier 1 tools (red) tend to have high precision but sometimes lower recall — the LLM
  is conservative about granting dangerous tools, which is the preferred failure mode.
- Tools showing recall gaps (recall significantly below precision) are where the LLM
  under-grants — these are the primary operational cost of the system.
- Tools with N/A recall are never granted in the sample; their true recall cannot
  be estimated at n=60 and will require the full experiment run.

In [ ]:
per_tool = post["permission_metrics"]["per_tool"]

# Sort: tier ascending, then F1 descending within tier
def sort_key(t):
    f1 = per_tool[t].get("f1") or 0
    return (TOOL_TIER[t], -f1)

sorted_tools = sorted(TOOLS, key=sort_key)
precisions   = [per_tool[t].get("precision") for t in sorted_tools]
recalls      = [per_tool[t].get("recall")    for t in sorted_tools]
f1s          = [per_tool[t].get("f1")        for t in sorted_tools]

# Replace None with 0 for plotting; track which are N/A
prec_plot = [p if p is not None else 0 for p in precisions]
rec_plot  = [r if r is not None else 0 for r in recalls]

y     = np.arange(len(sorted_tools))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 8))

bars_p = ax.barh(y - width / 2, prec_plot, width, alpha=0.88)
bars_r = ax.barh(y + width / 2, rec_plot,  width, alpha=0.45)

for bar, t in zip(bars_p, sorted_tools):
    bar.set_color(TIER_COLOR[TOOL_TIER[t]])
for bar, t in zip(bars_r, sorted_tools):
    bar.set_color(TIER_COLOR[TOOL_TIER[t]])

# Tier group dividers
current_tier = TOOL_TIER[sorted_tools[0]]
for i, t in enumerate(sorted_tools):
    if TOOL_TIER[t] != current_tier:
        ax.axhline(y=i - 0.5, color="#dddddd", linewidth=1.2)
        current_tier = TOOL_TIER[t]

# N/A annotations
for i, (p, r) in enumerate(zip(precisions, recalls)):
    if p is None:
        ax.text(0.01, i - width / 2, "N/A", va="center", fontsize=7.5,
                color=C["grey"], style="italic")
    if r is None:
        ax.text(0.01, i + width / 2, "N/A", va="center", fontsize=7.5,
                color=C["grey"], style="italic")

ax.set_yticks(y)
ax.set_yticklabels([f"[T{TOOL_TIER[t]}] {t}" for t in sorted_tools], fontsize=9)
ax.set_xlabel("Score (0–1)")
ax.set_xlim(0, 1.18)
ax.axvline(x=1.0, color="#eeeeee", linewidth=1)
ax.set_title("Per-tool precision & recall by tier (post-review)", fontweight="bold")

solid_patch = mpatches.Patch(color=C["grey"], alpha=0.88, label="Precision (solid)")
light_patch = mpatches.Patch(color=C["grey"], alpha=0.45, label="Recall (light)")
tier_patches = [
    mpatches.Patch(color=TIER_COLOR[k], label=TIER_LABEL[k]) for k in [1, 2, 3]
]
ax.legend(handles=[solid_patch, light_patch] + tier_patches,
          loc="lower right", fontsize=8)

plt.suptitle("Figure 3 — Per-tool precision & recall (post-review, n=60)",
             fontsize=11, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig(METRICS / "fig3_per_tool_metrics.pdf", bbox_inches="tight")
plt.show()
print("Saved → dataset/metrics/fig3_per_tool_metrics.pdf")

## 7. Figure 4 — Disagreement resolution breakdown

**What this shows:** How the 20 permission disagreements and 24 sensitivity
disagreements were resolved. Each bar is split into three segments: cases where
the LLM was correct, cases where the human was correct, and genuinely ambiguous cases.

**How to read it:** The width of each coloured segment corresponds to the count
of resolutions in that category. Percentages are shown inside each segment.

**Key findings:**
- **Permissions (top bar):** The LLM and human were wrong at nearly equal rates —
  50% LLM correct vs 40% human correct. This near-parity is why adjudicated
  post-review metrics are the primary claim: human labels are not a reliable
  enough ground truth to use as-is.
- **Sensitivity (bottom bar):** The human was correct far more often (70.8% vs 25%).
  The LLM has a genuine weakness in sensitivity classification. Combined with the
  60% overall agreement rate, this reinforces the decision to exclude sensitivity
  from all classifier metrics.
- The sensitivity bar is shown for context only — it does not contribute to any
  performance figure in the paper.

In [ ]:
rb   = post["resolution_breakdown"]
perm = rb["permissions"]
sens = rb["sensitivity"]

rows_data = [
    ("Sensitivity\n(n=24 — context only)", sens),
    ("Permissions\n(n=20)",                perm),
]

fig, ax = plt.subplots(figsize=(10, 3.2))

for i, (label, d) in enumerate(rows_data):
    total = d["llm_correct"] + d["human_correct"] + d["ambiguous"]
    left  = 0
    segments = [
        (d["llm_correct"],   C["blue"],    "LLM correct"),
        (d["human_correct"], C["orange"],  "Human correct"),
        (d["ambiguous"],     C["grey"],    "Ambiguous"),
    ]
    for val, color, _ in segments:
        if val == 0:
            left += val
            continue
        ax.barh(i, val, left=left, height=0.45, color=color, alpha=0.85)
        # Label inside segment
        if val / total > 0.07:
            ax.text(left + val / 2, i,
                    f"{val}\n({val/total*100:.0f}%)",
                    ha="center", va="center",
                    fontsize=9, fontweight="bold",
                    color="white" if color != C["grey"] else "black")
        left += val

ax.set_yticks(range(len(rows_data)))
ax.set_yticklabels([r[0] for r in rows_data], fontsize=9)
ax.set_xlabel("Number of disagreements resolved")
ax.set_xlim(0, 28)

# Dashed separator between permission and sensitivity rows
ax.axhline(y=0.5, color="#cccccc", linestyle="--", linewidth=0.8)

# Annotate sensitivity row as context only
ax.text(25.5, 0, "context\nonly", va="center", ha="left",
        fontsize=8, color=C["grey"], style="italic")

legend_handles = [
    mpatches.Patch(color=C["blue"],   label="LLM correct"),
    mpatches.Patch(color=C["orange"], label="Human correct"),
    mpatches.Patch(color=C["grey"],   label="Ambiguous"),
]
ax.legend(handles=legend_handles, loc="lower right", fontsize=8)
ax.set_title("Disagreement resolution breakdown", fontweight="bold")

plt.suptitle("Figure 4 — Resolution of human-LLM disagreements",
             fontsize=11, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(METRICS / "fig4_resolution_breakdown.pdf", bbox_inches="tight")
plt.show()
print("Saved → dataset/metrics/fig4_resolution_breakdown.pdf")

## 8. Figure 5 — Sensitivity confusion matrix

**What this shows:** How human sensitivity labels (rows) relate to LLM sensitivity
labels (columns). Each cell shows the count of records with that human/LLM combination.
The green squares highlight the diagonal — perfect agreement cells.
Darker blue cells = more records.

**How to read it:** A perfect classifier would have all records on the diagonal
(top-left to bottom-right). Off-diagonal cells are disagreements.
The direction of off-diagonal errors tells you whether the LLM over-classifies
or under-classifies relative to the human.

**Key findings:**
- The **HIGH row** is reasonably well-concentrated on the diagonal (24/34, 70.6%),
  but 9 HIGH records are called MEDIUM by the LLM. The LLM under-classifies tasks
  that the human considers high-sensitivity.
- The **MEDIUM row** is the most scattered: only 11/22 (50%) land on the diagonal,
  with 4 called LOW and 7 called HIGH. MEDIUM is genuinely unstable — errors go
  in both directions, not just one, which indicates definitional ambiguity rather
  than a consistent bias.
- The **LOW row** has too few records (n=4) to draw conclusions.
- This pattern supports the decision to **exclude sensitivity from all metric
  computations**: the disagreements reflect different definitions of the tiers,
  not a learnable signal the classifier could improve on with more data.

In [ ]:
cm_raw = pre["sensitivity_metrics"]["confusion_matrix"]
matrix = np.array([
    [cm_raw.get(f"human_{h}_llm_{l}", 0) for l in SENSITIVITIES]
    for h in SENSITIVITIES
])

row_totals = matrix.sum(axis=1)

fig, ax = plt.subplots(figsize=(6.5, 5.5))

im = ax.imshow(matrix, cmap="Blues", aspect="auto",
               vmin=0, vmax=matrix.max())

# Cell annotations
for i in range(3):
    for j in range(3):
        val  = matrix[i, j]
        pct  = val / row_totals[i] * 100 if row_totals[i] > 0 else 0
        tc   = "white" if val > matrix.max() * 0.55 else "#333333"
        ax.text(j, i, f"{val}\n({pct:.0f}%)",
                ha="center", va="center",
                fontsize=11, fontweight="bold", color=tc)

# Highlight diagonal
for i in range(3):
    ax.add_patch(plt.Rectangle(
        (i - 0.5, i - 0.5), 1, 1,
        fill=False,
        edgecolor=C["green"], linewidth=2.5
    ))

ax.set_xticks(range(3))
ax.set_yticks(range(3))
ax.set_xticklabels([f"LLM: {s}" for s in SENSITIVITIES], fontsize=10)
ax.set_yticklabels([f"Human: {s}  (n={int(t)})" for s, t in zip(SENSITIVITIES, row_totals)],
                   fontsize=10)
ax.set_xlabel("LLM prediction", fontsize=10)
ax.set_ylabel("Human label", fontsize=10)
ax.set_title("Sensitivity tier confusion matrix", fontweight="bold", fontsize=11)

plt.colorbar(im, ax=ax, shrink=0.75, label="Count")

# Agreement rate annotation
agree_rate = pre["sensitivity_metrics"]["agreement_rate"]
ax.text(2.6, -0.6, f"Overall agreement: {agree_rate:.1%}",
        ha="right", fontsize=9, color=C["grey"])

plt.suptitle("Figure 5 — Sensitivity confusion matrix (n=60, excluded from metrics)",
             fontsize=11, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig(METRICS / "fig5_sensitivity_confusion.pdf", bbox_inches="tight")
plt.show()
print("Saved → dataset/metrics/fig5_sensitivity_confusion.pdf")

## 9. Key findings summary

A concise record of every number that goes into the paper.

In [ ]:
rb   = post["resolution_breakdown"]
perm = rb["permissions"]
sens = rb["sensitivity"]

print("═" * 60)
print("  MOSTARGATE — HUMAN VALIDATION SUMMARY")
print("═" * 60)

print("\nDATASET")
print(f"  Total records:              {len(dataset)}")
print(f"  Human-validated sample:     {len(human_results)}  (10%)")
print(f"  Avg permissions/record:     {avg_perms:.2f}")

print("\nPERMISSION METRICS — PRE-REVIEW  (human = reference)")
print(f"  Exact match rate:           {pre_ov['exact_match_rate']:.1%}")
print(f"  Cohen's kappa:              {pre_ov['kappa']:.3f}")
print(f"  Overshoot rate:             {pre_ov['overshoot_rate']:.1%}")
print(f"  Undershoot rate:            {pre_ov['undershoot_rate']:.1%}")
print(f"  Severity-weighted overshoot:{pre_ov['severity_weighted_overshoot']:.2f}")

print("\nPERMISSION METRICS — POST-REVIEW  (adjudicated ground truth)")
print(f"  Exact match rate:           {post_ov['exact_match_rate']:.1%}  ← primary claim")
print(f"  Cohen's kappa:              {post_ov['kappa']:.3f}      ← primary claim")
print(f"  Overshoot rate:             {post_ov['overshoot_rate']:.1%}")
print(f"  Undershoot rate:            {post_ov['undershoot_rate']:.1%}  ← primary failure mode")
print(f"  Severity-weighted overshoot:{post_ov['severity_weighted_overshoot']:.2f}      ← headline security result")

total_p = perm['llm_correct'] + perm['human_correct'] + perm['ambiguous']
print("\nDISAGREEMENT RESOLUTION  (permissions, n=20)")
print(f"  LLM correct:   {perm['llm_correct']}/{total_p}  ({perm['llm_correct']/total_p:.0%}) — human labelling errors")
print(f"  Human correct: {perm['human_correct']}/{total_p}  ({perm['human_correct']/total_p:.0%}) — LLM prediction errors")
print(f"  Ambiguous:     {perm['ambiguous']}/{total_p}  ({perm['ambiguous']/total_p:.0%})")

total_s = sens['llm_correct'] + sens['human_correct'] + sens['ambiguous']
print("\nSENSITIVITY  (excluded from all metrics — shown for context)")
print(f"  Agreement rate:  {pre['sensitivity_metrics']['agreement_rate']:.1%}")
print(f"  Human correct:   {sens['human_correct']}/{total_s}  ({sens['human_correct']/total_s:.0%})")
print(f"  LLM correct:     {sens['llm_correct']}/{total_s}  ({sens['llm_correct']/total_s:.0%})")

print("\nFIGURES SAVED")
for f in sorted(METRICS.glob("fig*.pdf")):
    print(f"  {f.relative_to(ROOT)}")

print("\n═" * 60)